# s99 MDD facts, tables and figures (Step 3)

READ ONLY. Builds everything the MDD may quote from the Step 1 evidence workbooks only (`<doc_output_root>/<output_folder>/evidence/mdd_evidence_<key>.xlsx`): `mdd_facts.json` (flat dictionary `<model key>|shared|diff.<area>.<item>` -> value, display, source_file, source_sheet, source_cell_or_row), `tables/<key>.csv` + sidecar json, `figures/<name>.png` + caption/source json, and `facts_gaps.csv` (shared facts that differ between workbooks, sheets missing, and content requested by the approved template map that the evidence does not supply). No metric is recomputed: every value is copied from an evidence cell and re-checked against it; figures are drawn from the written tables and re-checked against the evidence. Writes only under `<doc_output_root>/<output_folder>/` (run summary in `run_summaries/s99_mdd_facts.json`; no MLflow run, as for the other s99 modules).

In [ ]:
# The project has no requirements.lock yet (same note as the other s99 modules); this module needs openpyxl + pyyaml + matplotlib.
%pip install openpyxl pyyaml matplotlib --quiet
dbutils.library.restartPython()

In [ ]:
# s99_mdd_facts: READ ONLY. Reads only the Step 1 evidence workbooks, the approved template map and config;
# writes only under <doc_output_root>/<output_folder>/ (mdd_facts.json, facts_gaps.csv, tables/, figures/, run_summaries/).
import sys, os, re, json, math, hashlib, importlib, random, shutil
from datetime import datetime, timezone, date
from collections import Counter, OrderedDict
sys.path.insert(0, os.getcwd())            # notebook lives in src/
import common
importlib.reload(common)
import numpy as np, pandas as pd, yaml, openpyxl
from openpyxl.utils import get_column_letter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, FuncFormatter

MODULE = "s99_mdd_facts"
PROJECT_ROOT = os.path.dirname(os.getcwd())
CFG_PATH = os.path.join(PROJECT_ROOT, "config", "config.yaml")
READS = []                                   # every input file opened (acceptance: config + evidence + map only)
def note_read(p):
    READS.append(os.path.realpath(p))
    return p
with open(note_read(CFG_PATH)) as f:
    cfg = yaml.safe_load(f)
SEED = cfg["random_seed"]
random.seed(SEED); np.random.seed(SEED)
MDD = cfg["mdd"]
RUN_TIME = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def git_info():
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        oid = w.workspace.get_status(PROJECT_ROOT.replace("/Workspace", "", 1)).object_id
        r = w.repos.get(oid)
        return r.head_commit_id or "unknown", r.branch or "unknown"
    except Exception:
        return "unknown", "unknown"

CFG_SHA = sha256_file(CFG_PATH)
GIT, BRANCH = git_info()
MDDR = os.path.join(MDD["doc_output_root"], MDD["output_folder"])
EVID = os.path.join(MDDR, "evidence")
OUT_FACTS = os.path.join(MDDR, "mdd_facts.json")
OUT_GAPS = os.path.join(MDDR, "facts_gaps.csv")
TBL_DIR, FIG_DIR = os.path.join(MDDR, "tables"), os.path.join(MDDR, "figures")
RUN_SUM = os.path.join(MDDR, "run_summaries", f"{MODULE}.json")
MAP_APPROVED = os.path.join(MDDR, "template_map_approved.xlsx")
MAP_PATH = MAP_APPROVED if os.path.exists(MAP_APPROVED) else os.path.join(MDDR, "template_map.xlsx")
MAP_IS_APPROVED = MAP_PATH == MAP_APPROVED
MODELS = MDD["models"]
MKEYS = [m["key"] for m in MODELS]
DOC_MODEL = {m["key"]: str(m["model"]) for m in MODELS}
LABEL = {m["key"]: m["label"] for m in MODELS}
BENCH_A = {k: k.rsplit("_", 1)[0] + "_a" for k in MKEYS}     # Model A row key in the benchmark report (e.g. nofts_a)
EV_WB = {k: os.path.join(EVID, f"mdd_evidence_{k}.xlsx") for k in MKEYS}
for p in list(EV_WB.values()) + [MAP_PATH]:
    assert os.path.exists(p), f"input not found: {p}"

WATCH = [MDDR, os.path.join(PROJECT_ROOT, "config"), os.path.join(PROJECT_ROOT, "docs")]
def snapshot(roots):
    out = {}
    for r in roots:
        for dp, dn, fn in os.walk(r):
            for f in fn:
                p = os.path.join(dp, f)
                out[p] = sha256_file(p)
    return out
SNAP_BEFORE = snapshot(WATCH)
INPUT_SHA = {"config": CFG_SHA, "map": sha256_file(MAP_PATH), **{f"evidence_{k}": sha256_file(v) for k, v in EV_WB.items()}}

print(f"Module  : {MODULE} | run {RUN_TIME} UTC | branch {BRANCH} | git {GIT[:12]} | config {CFG_SHA[:12]}")
for k in MKEYS:
    print(f"Evidence: {k} model {DOC_MODEL[k]} | {EV_WB[k]}")
print(f"Map     : {MAP_PATH} (approved file present: {MAP_IS_APPROVED})")
print(f"Outputs : {MDDR}/ mdd_facts.json, facts_gaps.csv, tables/, figures/, run_summaries/{MODULE}.json")
print(f"Read-only snapshot: {len(SNAP_BEFORE)} files under {WATCH}")

In [ ]:
def native(v):
    """JSON-safe copy of a cell value (no rounding)."""
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (float, np.floating)):
        v = float(v)
        return None if math.isnan(v) else v
    if isinstance(v, (datetime, date)):
        return v.isoformat()
    return v

def load_raw(k):
    wb = openpyxl.load_workbook(note_read(EV_WB[k]), read_only=True, data_only=True)
    raw = {s: [list(r) for r in wb[s].iter_rows(values_only=True)] for s in wb.sheetnames}
    wb.close()
    return raw
RAW = {k: load_raw(k) for k in MKEYS}

def is_blank(r):
    return (not r) or all(v is None or (isinstance(v, str) and not v.strip()) for v in r)

def copied(rows):
    return bool(rows and rows[0] and isinstance(rows[0][0], str) and rows[0][0].startswith("SOURCE"))

def sheet_blocks(rows):
    """Contiguous non-blank row blocks as (first Excel row, rows). Copied sheets start after the 3 source lines."""
    start = 3 if copied(rows) else 0
    out, cur = [], None
    for i, r in enumerate(rows[start:], start + 1):
        if is_blank(r):
            if cur:
                out.append(cur); cur = None
            continue
        if cur is None:
            cur = [i, []]
        cur[1].append(r)
    if cur:
        out.append(cur)
    return out

def make_tab(k, s, header=True):
    """First block with >= 2 rows and >= 2 filled cells in its first row; a one-cell title row is skipped."""
    rows_all = RAW[k][s]
    for r0, rows in sheet_blocks(rows_all):
        title = None
        if len(rows) > 2 and sum(v is not None for v in rows[0]) == 1:
            title, rows, r0 = rows[0][0], rows[1:], r0 + 1
        if len(rows) >= 2 and sum(v is not None for v in rows[0]) >= 2:
            break
    else:
        return None
    ncol = max(len(r) for r in rows)
    rows = [list(r) + [None] * (ncol - len(r)) for r in rows]
    if header:
        hdr = [str(v).strip() if v is not None else f"col{j + 1}" for j, v in enumerate(rows[0])]
        data, first = rows[1:], r0 + 1
    else:
        hdr, data, first = [f"col{j + 1}" for j in range(ncol)], rows, r0
    keep = [j for j in range(ncol) if (header and rows[0][j] is not None) or any(r[j] is not None for r in data)]
    names = [hdr[j] for j in keep]
    assert len(set(names)) == len(names), f"{k}/{s}: duplicate column names {names}"
    df = pd.DataFrame([[r[j] for j in keep] for r in data], columns=names)
    return {"k": k, "sheet": s, "df": df, "rows": list(range(first, first + len(data))),
            "cols": {hdr[j]: get_column_letter(j + 1) for j in keep}, "header_row": r0 if header else None, "title": title,
            "origin": rows_all[0][0] if copied(rows_all) else "evidence workbook (structural sheet)", "file": EV_WB[k]}

STATUS = {}
for k in MKEYS:
    t = make_tab(k, "index")
    STATUS[k] = dict(zip(t["df"]["sheet"], t["df"]["status"]))

def pick_sheet(k, expr):
    """'a|b' = either sheet (model-specific names, e.g. ri_summary | ri_fts_summary)."""
    return next((a for a in expr.split("|") if a in RAW[k]), None)

for k in MKEYS:
    print(f"{k}: {len(RAW[k])} sheets | index status {dict(Counter(STATUS[k].values()))}")

In [ ]:
FACTS, GAPS, NOTES = OrderedDict(), [], []
USED = {k: set() for k in MKEYS}

def gap(key, model_key, section, gap_type, detail, requested_by=""):
    GAPS.append({"key": key, "model_key": model_key, "section": section, "gap_type": gap_type, "detail": str(detail)[:500],
                 "requested_by": requested_by})

def slug(v):
    return re.sub(r"[^0-9a-z]+", "_", str(v).lower()).strip("_") or "blank"

def rate_tag(v):
    return "p" + f"{float(v) * 100:g}".replace(".", "_")

REN = {"gini_lo": "gini_ci_low", "gini_hi": "gini_ci_high", "auc_lo": "auc_ci_low", "auc_hi": "auc_ci_high",
       "ks_lo": "ks_ci_low", "ks_hi": "ks_ci_high", "net_bads_avoided": "net_bads"}
def rn(c):
    return REN.get(c, slug(c))

def M(k, model, rest):
    """Documented model (config.mdd model, e.g. C) without prefix; other models as model_<x>_..."""
    return rest if str(model) == DOC_MODEL[k] else f"model_{slug(model)}_{rest}"

# display rules (formatting only, value unchanged): counts with thousands separators, rates as % with 1 decimal,
# Gini / AUC / KS / PSI / IV 3 decimals, p-values 3 decimals (<0.001), VIF 2 decimals, other decimals 4 places
P_RE = re.compile(r"(^|[._])(p|p_le_0|p_delta_le_0|p_value|p_two_sided|pvalue)($|[._])")
PCT_T = {"rate", "share", "capture", "pct", "approval", "pd"}
G3_T = {"gini", "auc", "ks", "dgini", "dauc", "dks", "psi", "iv"}
def fmt(key, v):
    if v is None:
        return ""
    if isinstance(v, bool):
        return "Yes" if v else "No"
    if isinstance(v, str):
        sv = v.strip()
        return f"{int(sv):,}" if re.fullmatch(r"-?\d+", sv) else sv
    t = set(re.split(r"[._]", key.lower()))
    if P_RE.search(key.lower()):
        return "<0.001" if v < 0.001 else f"{v:.3f}"
    if "learning" in t:
        return f"{v:.4f}"
    if t & PCT_T or any(x.startswith("top") and x[3:].isdigit() for x in t):
        return f"{v * 100:.1f}%"
    if isinstance(v, int):
        return f"{v:,}"
    if t & G3_T:
        return f"{v:.3f}"
    if "vif" in t:
        return f"{v:.2f}"
    if float(v).is_integer():
        return f"{int(v):,}"
    return f"{v:.4f}"

def src_of(tab, i, col):
    return {"source_file": tab["file"], "source_sheet": tab["sheet"], "source_cell_or_row": f"{tab['cols'][col]}{tab['rows'][i]}",
            "origin": tab["origin"]}

def put(key, v, src, extra=None):
    if key in FACTS and FACTS[key]["value"] != v:
        new = f"{key}_row{re.sub(r'^[A-Z]+', '', src['source_cell_or_row'])}"
        NOTES.append(f"duplicate key {key} with a different value -> {new}")
        key = new
    if key not in FACTS:
        FACTS[key] = {"value": v, "display": fmt(key, v), **src, **(extra or {})}
    return key

def S(sheet, scope, area, keys=None, vals=None, item=None, kv=False, header=True, aliases=False):
    return dict(sheet=sheet, scope=scope, area=area, keys=keys, vals=vals, item=item, kv=kv, header=header, aliases=aliases)

def col_item(*cols):
    return lambda r, c, k: "_".join([slug(r[x]) for x in cols] + [rn(c)])

def bench1(r, c, k):
    sc, base = str(r["score"]), f"{slug(r['view'])}_{slug(r['period'])}_{rn(c)}"
    if sc == k:
        return ("model", base)
    if sc == BENCH_A[k]:
        return ("model", f"model_a_{base}")
    if sc in MKEYS or sc in BENCH_A.values():
        return None
    return ("shared", f"{slug(sc)}_{base}")

def bench2(r, c, k):
    x, y = str(r["x"]), str(r["y"])
    tail = f"{slug(r['metric'])}_{rn(c)}_{slug(r['period'])}"
    if {x, y} == set(MKEYS):
        return ("diff", f"{slug(x)}_vs_{slug(y)}_{tail}")
    if x == k:
        return ("model", f"vs_{slug(y)}_{tail}")
    if x in MKEYS:
        return None
    return ("shared", f"{slug(x)}_vs_{slug(y)}_{tail}")

CUT_COLS = ["cutoff_score", "approval_rate_achieved", "n_approved", "n_declined", "bads_approved", "bads_declined",
            "share_of_bads_declined", "approved_bad_rate", "approved_bad_rate_ci_low", "approved_bad_rate_ci_high", "declined_bad_rate"]
CVA_COLS = ["cutoff_C", "cutoff_A", "approval_rate_C", "approval_rate_A", "bads_declined_C", "bads_declined_A", "net_bads_avoided_by_C",
            "net_ci_low", "net_ci_high", "p_delta_le_0", "approved_bad_rate_C", "approved_bad_rate_A"]
FACT_SPECS = [
    # shared: data, counts, windows (must be identical in both workbooks)
    S("data_filter_waterfall", "shared", "waterfall", ["step"], None, col_item("step")),
    S("data_confirmed_counts_cfg", "shared", "confirmed_counts", kv=True),
    S("data_counts_by_window", "shared", "counts", ["window"], None, col_item("window")),
    S("data_dev_early_late", "shared", "counts", ["segment"], None, col_item("segment")),
    S("data_status_by_window", "shared", "status", ["window"], None, col_item("window")),
    S("data_tfold_stats", "shared", "tfold", ["fold", "part"], None, lambda r, c, k: f"fold_{slug(r['fold'])}_{slug(r['part'])}_{rn(c)}"),
    S("data_active_reporting_xtab", "shared", "active_reporting", ["AR_norm"], None, col_item("AR_norm")),
    S("oot_counts", "shared", "oot_counts", ["sample", "period"], None, col_item("sample", "period")),
    S("L2_periods", "shared", "l2_periods", ["period"], None, col_item("period")),
    # model-specific
    S("meta", "model", "meta", [], None, lambda r, c, k: slug(c)),
    S("registry_fields", "model", "registry", kv=True, header=False),
    S("selection_override", "model", "selection", kv=True, header=False),
    S("fun_by_layer", "model", "funnel", ["source", "layer"], None, col_item("source", "layer")),
    S("fun_by_source", "model", "funnel_source", ["source"], None, col_item("source")),
    S("fun_drop_rules", "model", "funnel_drop", ["dropped_at", "drop_rule"], ["n"], lambda r, c, k: f"{slug(r['dropped_at'])}_{slug(r['drop_rule'])}"),
    S("L0_counts_by_reason", "model", "l0", ["L0_reason"], ["count"], lambda r, c, k: slug(r["L0_reason"])),
    S("L1_counts_by_reason", "model", "l1", ["L1_reason"], ["count"], lambda r, c, k: slug(r["L1_reason"])),
    S("L3_red_features", "model", "psi_red", ["feature"], ["cause_type", "decision"], col_item("feature")),
    S("L5_selection_info", "model", "l5", kv=True),
    S("mod_tuning_choice", "model", "tuning", ["model"], ["trial", "cv_auc_mean", "t_auc_mean", "final_rounds"], lambda r, c, k: M(k, r["model"], rn(c))),
    S("mod_fixed_params", "model", "fixed_params", ["model", "param"], ["value"], lambda r, c, k: M(k, r["model"], slug(r["param"]))),
    S("mod_chosen_params", "model", "chosen_params", kv=True),
    S("mod_lgb_rounds", "model", "lgb_rounds", kv=True),
    S("mod_oof_metrics", "model", "dev_oof", ["model"], None, lambda r, c, k: M(k, r["model"], rn(c))),
    S("mod_tfold_summary", "model", "tfold", ["model"], None, lambda r, c, k: M(k, r["model"], rn(c))),
    S("oot_summary", "model", "oot_summary", kv=True),
    S("oot_run_summary", "model", "oot_run", kv=True),
    S("oot_metrics", "model", "oot", ["model", "period"], None, lambda r, c, k: M(k, r["model"], f"{rn(c)}_{slug(r['period'])}"), aliases=True),
    S("oot_paired_bootstrap", "model", "oot_boot", ["pair", "period", "metric"], ["mean", "ci_low", "ci_high", "p_le_0"],
      lambda r, c, k: f"{slug(r['pair'])}_{slug(r['metric'])}_{rn(c)}_{slug(r['period'])}"),
    S("oot_delong", "model", "oot_delong", ["pair", "period"], None, lambda r, c, k: f"{slug(r['pair'])}_{rn(c)}_{slug(r['period'])}"),
    S("oot_rules_R1_R4", "model", "rules", ["challenger", "rule"], ["result", "reason"], col_item("challenger", "rule")),
    S("oot_calibration", "model", "calibration", ["model", "period"], None, lambda r, c, k: M(k, r["model"], f"{rn(c)}_{slug(r['period'])}"), aliases=True),
    S("oot_score_psi", "model", "score_psi", ["model", "period"], None, lambda r, c, k: M(k, r["model"], f"{rn(c)}_{slug(r['period'])}"), aliases=True),
    S("oot_band_monotonic", "model", "band_monotonic", ["model", "kind", "period"], ["monotonic", "n_reversals"],
      lambda r, c, k: M(k, r["model"], f"{slug(r['kind'])}_{slug(r['period'])}_{rn(c)}")),
    S("cal_platt", "model", "platt", ["model"], None, lambda r, c, k: M(k, r["model"], rn(c))),
    S("cal_platt_file", "model", "platt_file", kv=True),
    S("cal_scaling_A", "model", "scaling_a", kv=True),
    S("cal_factor_offset_A", "model", "scaling_a", kv=True),
    S("cut_grid_oot_C", "model", "cutoff", ["approval_rate_target"], CUT_COLS, lambda r, c, k: f"oot.{rate_tag(r['approval_rate_target'])}.{rn(c)}"),
    S("cut_grid_oot_A", "model", "cutoff", ["approval_rate_target"], CUT_COLS, lambda r, c, k: f"oot.model_a.{rate_tag(r['approval_rate_target'])}.{rn(c)}"),
    S("cut_grid_dev_oof_C", "model", "cutoff", ["approval_rate_target"], CUT_COLS, lambda r, c, k: f"dev_oof.{rate_tag(r['approval_rate_target'])}.{rn(c)}"),
    S("cut_grid_dev_oof_A", "model", "cutoff", ["approval_rate_target"], CUT_COLS, lambda r, c, k: f"dev_oof.model_a.{rate_tag(r['approval_rate_target'])}.{rn(c)}"),
    S("cut_C_vs_A", "model", "cutoff_c_vs_a", ["approval_rate_target"], CVA_COLS, lambda r, c, k: f"{rate_tag(r['approval_rate_target'])}.{rn(c)}"),
    S("cut_summary", "model", "cutoff_summary", ["approval_rate_target"], None, lambda r, c, k: f"{rate_tag(r['approval_rate_target'])}.{rn(c)}"),
    S("benchmark_step1_discrimination", "auto", "benchmark", ["view", "period", "score", "label", "role"],
      ["n", "n_bad", "bad_rate", "auc", "auc_lo", "auc_hi", "gini", "gini_lo", "gini_hi", "ks", "ks_lo", "ks_hi", "top10_capture", "top20_capture"], bench1),
    S("benchmark_step2_paired_deltas", "auto", "benchmark_delta", ["period", "x", "y", "metric"],
      ["x_value", "y_value", "delta_observed", "ci_low", "ci_high", "p_delta_le_0"], bench2),
    S("ri_summary|ri_fts_summary", "model", "ri", None, ["value", "detail"],
      lambda r, c, k: "_".join([slug(r[x]) for x in r if x not in ("value", "detail") and r[x] is not None] + ([] if c == "value" else [c]))),
    # FTS vs no-FTS comparison reports (copied into both workbooks; must be identical)
    S("fts_cmp_performance", "diff", "fts_cmp", ["model"], None, col_item("model")),
    S("fts_cmp_paired_bootstrap", "diff", "fts_cmp", ["pair", "metric"], ["mean", "ci_low", "ci_high", "p_le_0"], lambda r, c, k: f"{slug(r['metric'])}_{rn(c)}"),
    S("fts_cmp_counts", "diff", "fts_cmp_counts", ["step"], ["n"], lambda r, c, k: slug(r["step"])),
    S("fts_cmp_net_bads_ci", "diff", "cutoff", ["population", "pair", "approval_rate"],
      ["declined_each", "baseline_rejected_bads", "challenger_rejected_bads", "net_bads_avoided", "boot_mean", "boot_se", "ci_low", "ci_high", "p_le_0", "significant"],
      lambda r, c, k: f"{rate_tag(r['approval_rate'])}.{rn(c)}"),
]

def spec_rows(spec, k, tab):
    df, out = tab["df"], []
    if spec["kv"]:
        fcol, vcol = df.columns[0], df.columns[1]
        for i, r in df.iterrows():
            field, v = native(r[fcol]), native(r[vcol])
            if field is None or re.search(r"sha|hash", str(field), re.I) or (isinstance(v, str) and re.fullmatch(r"[0-9a-f]{40,64}", v)):
                continue
            out.append((spec["scope"], slug(field), v, src_of(tab, i, vcol)))
        return out
    keys = spec["keys"] if spec["keys"] is not None else [c for c in df.columns if c not in (spec["vals"] or [])]
    vals = [c for c in (spec["vals"] or [c for c in df.columns if c not in keys]) if c in df.columns]
    for c in [c for c in (spec["vals"] or []) if c not in df.columns and c != "detail"]:
        gap(f"{k}.{spec['area']}", k, "", "COLUMN_MISSING", f"{tab['sheet']}: column {c} not in the evidence sheet")
    for i, r in df.iterrows():
        rd = {c: native(r[c]) for c in df.columns}
        for c in vals:
            res = spec["item"](rd, c, k)
            if res is None:
                continue
            scope, item = res if isinstance(res, tuple) else (spec["scope"], res)
            out.append((scope, item, rd[c], src_of(tab, i, c)))
    return out

SHARED = {k: OrderedDict() for k in MKEYS}
for spec in FACT_SPECS:
    for k in MKEYS:
        s = pick_sheet(k, spec["sheet"])
        if s is None:
            gap(f"{k}.{spec['area']}", k, "", "SHEET_MISSING", f"evidence sheet {spec['sheet']} not in the {k} workbook")
            continue
        tab = make_tab(k, s, header=spec["header"])
        if tab is None or tab["df"].empty:
            gap(f"{k}.{spec['area']}", k, "", "EMPTY", f"evidence sheet {s} has no rows")
            continue
        USED[k].add(s)
        for scope, item, v, src in spec_rows(spec, k, tab):
            if scope == "model":
                key = put(f"{k}.{spec['area']}.{item}", v, src)
                if spec["aliases"] and item.endswith("_overall"):
                    put(f"{k}.{spec['area']}.{item[:-8]}", v, src, {"alias_of": key})
            else:
                kk = (scope, spec["area"], item)
                if kk in SHARED[k] and SHARED[k][kk][0] != v:
                    kk = (scope, spec["area"], f"{item}_row{re.sub(r'^[A-Z]+', '', src['source_cell_or_row'])}")
                SHARED[k].setdefault(kk, (v, src))

# shared / diff facts: identical in every workbook -> one key; otherwise facts_gaps + per-model keys
for kk in OrderedDict((x, None) for k in MKEYS for x in SHARED[k]):
    scope, area, item = kk
    got = {k: SHARED[k].get(kk) for k in MKEYS}
    vals = {json.dumps(g[0]) for g in got.values() if g is not None}
    if all(g is not None for g in got.values()) and len(vals) == 1:
        v, src = got[MKEYS[0]]
        put(f"{scope}.{area}.{item}", v, src, {"also_in": [{x: got[k][1][x] for x in ("source_file", "source_sheet", "source_cell_or_row")}
                                                         for k in MKEYS[1:]]})
    else:
        gap(f"{scope}.{area}.{item}", ",".join(k for k, g in got.items() if g), "", "SHARED_MISMATCH",
            "; ".join(f"{k}={g[0]!r} @ {g[1]['source_sheet']}!{g[1]['source_cell_or_row']}" if g else f"{k}=missing" for k, g in got.items()))
        for k, g in got.items():
            if g:
                put(f"{k}.{area}.{item}", g[0], g[1], {"note": f"{scope} fact not identical in both workbooks; kept per model"})

print(f"{len(FACTS)} facts | by prefix {dict(Counter(k.split('.')[0] for k in FACTS))} | gaps so far {len(GAPS)} | notes {len(NOTES)}")
for key in ["shared.counts.dev_records", "fts_c.oot.gini_overall", "nofts_c.oot.gini_ci_low", "diff.cutoff.p95.net_bads"]:
    e = FACTS.get(key)
    print(f"  {key}: " + (f"{e['value']!r} -> {e['display']} | {e['source_sheet']}!{e['source_cell_or_row']}" if e else "MISSING"))

In [ ]:
TABLE_SPECS = [  # (name, evidence sheet, scope)
    ("waterfall", "data_filter_waterfall", "shared"), ("counts_by_window", "data_counts_by_window", "shared"),
    ("status_by_window", "data_status_by_window", "shared"), ("dev_early_late", "data_dev_early_late", "shared"),
    ("vintage_monthly", "target_vintage_monthly", "shared"), ("tfold_stats", "data_tfold_stats", "shared"),
    ("active_reporting_xtab", "data_active_reporting_xtab", "shared"), ("confirmed_counts_cfg", "data_confirmed_counts_cfg", "shared"),
    ("oot_counts", "oot_counts", "shared"), ("l2_periods", "L2_periods", "shared"),
    ("benchmark_paired_deltas", "benchmark_step2_paired_deltas", "shared"),
    ("fts_cmp_performance", "fts_cmp_performance", "diff"), ("fts_cmp_paired_bootstrap", "fts_cmp_paired_bootstrap", "diff"),
    ("fts_cmp_counts", "fts_cmp_counts", "diff"), ("fts_cmp_net_bads_ci", "fts_cmp_net_bads_ci", "diff"),
] + [(n, s, "model") for n, s in [
    ("funnel_by_layer", "fun_by_layer"), ("funnel_by_source", "fun_by_source"), ("funnel_by_layer_source", "fun_by_layer_source"),
    ("funnel_drop_rules", "fun_drop_rules"), ("l0_counts_by_reason", "L0_counts_by_reason"), ("l0_counts_by_source", "L0_counts_by_source"),
    ("l0_feature_detail", "L0_feature_detail"), ("l1_counts_by_reason", "L1_counts_by_reason"), ("l1_iv_table", "L1_iv_table"),
    ("l2_period_results", "L2_period_results"), ("psi_summary", "L3_psi_summary"), ("psi_red_features", "L3_red_features"),
    ("psi_decisions", "L3_psi_decisions_file"), ("l4_pool_summary", "L4_pool_summary"), ("l4_vif", "L4_vif"), ("l4_corr_drops", "L4_corr_drops"),
    ("l5_final_list", "L5_final_list"), ("tuning_choice", "mod_tuning_choice"), ("fixed_params", "mod_fixed_params"),
    ("c_shortlist", "mod_C_shortlist"), ("c_tfold", "mod_C_tfold"), ("a_points", "mod_A_points"), ("a_coefficients", "mod_A_coefficients"),
    ("c_importance", "mod_C_importance"), ("oof_metrics", "mod_oof_metrics"), ("oof_fold_metrics", "mod_oof_fold_metrics"),
    ("tfold_metrics", "mod_tfold_metrics"), ("tfold_summary", "mod_tfold_summary"), ("oot_metrics", "oot_metrics"),
    ("oot_paired_bootstrap", "oot_paired_bootstrap"), ("oot_delong", "oot_delong"), ("oot_rules", "oot_rules_R1_R4"),
    ("oot_bands", "oot_bands"), ("oot_band_monotonic", "oot_band_monotonic"), ("oot_deciles", "oot_decile_csv"),
    ("oot_calibration", "oot_calibration"), ("oot_calibration_by_decile", "oot_calibration_by_decile"), ("oot_score_psi", "oot_score_psi"),
    ("oot_segments", "oot_segments"), ("oot_watch_approval", "oot_watch_approval"), ("oot_l6_summary", "oot_L6_summary"),
    ("oot_diagnosis", "oot_diagnosis"), ("platt", "cal_platt"), ("cutoff_summary", "cut_summary"),
    ("cutoff_grid_oot_c", "cut_grid_oot_C"), ("cutoff_grid_oot_a", "cut_grid_oot_A"), ("cutoff_grid_dev_oof_c", "cut_grid_dev_oof_C"),
    ("cutoff_grid_dev_oof_a", "cut_grid_dev_oof_A"), ("cutoff_c_vs_a", "cut_C_vs_A"), ("cutoff_score_bands_c", "cut_score_bands_C"),
    ("benchmark_discrimination", "benchmark_step1_discrimination"), ("benchmark_deciles", "benchmark_step3_deciles"),
    ("benchmark_capture", "benchmark_step3_capture"), ("benchmark_segments", "benchmark_step5_segments"),
    ("benchmark_segment_deltas", "benchmark_step5_segment_deltas"), ("benchmark_roc_curve", "benchmark_step6_roc_curve"),
    ("benchmark_capture_curve", "benchmark_step6_capture_curve"), ("ri_summary", "ri_summary|ri_fts_summary"),
    ("ri_waterfall", "ri_intake_waterfall|ri_fts_universe_counts"), ("reason_code_map", "rc_map_used"), ("evidence_gaps", "gaps")]]

def norm_cell(v):
    v = native(v)
    if v is None:
        return ""
    if isinstance(v, bool):
        return str(v)
    try:
        return f"{float(str(v).strip()):.12g}"
    except ValueError:
        return str(v).strip()

def norm_rows(df):
    return [[norm_cell(v) for v in row] for row in df.itertuples(index=False)]

shutil.rmtree(TBL_DIR, ignore_errors=True)       # this module owns tables/; re-running overwrites it
os.makedirs(TBL_DIR)
TABLES = OrderedDict()

def write_table(key, tab, extra):
    df = tab["df"].apply(lambda col: col.map(native))
    path = os.path.join(TBL_DIR, f"{key}.csv")
    df.to_csv(path, index=False)
    side = {"table": key, "source_file": tab["file"], "source_sheet": tab["sheet"],
            "source_cell_or_row": f"rows {tab['rows'][0]}-{tab['rows'][-1]} (header row {tab['header_row']})",
            "columns": {c: tab["cols"][c] for c in df.columns}, "n_rows": len(df), "title": tab["title"], "origin": tab["origin"], **extra}
    with open(path[:-4] + ".json", "w") as f:
        json.dump(side, f, indent=1, ensure_ascii=False)
    TABLES[key] = {"tab": tab, "csv": path, "sidecar": path[:-4] + ".json"}

for name, sheet, scope in TABLE_SPECS:
    tabs = {}
    for k in MKEYS:
        s = pick_sheet(k, sheet)
        tkey = f"{k}.{name}" if scope == "model" else f"{scope}.{name}"
        if s is None:
            gap(tkey, k, "", "SHEET_MISSING", f"table {name}: evidence sheet {sheet} not in the {k} workbook")
            continue
        t = make_tab(k, s)
        if t is None or t["df"].empty:
            gap(tkey, k, "", "EMPTY", f"table {name}: evidence sheet {s} has no rows")
            continue
        tabs[k] = t
        USED[k].add(s)
    if scope == "model":
        for k, t in tabs.items():
            write_table(f"{k}.{name}", t, {})
    elif tabs:
        t0 = tabs[MKEYS[0]] if MKEYS[0] in tabs else None
        same = t0 is not None and len(tabs) == len(MKEYS) and all(
            list(t["df"].columns) == list(t0["df"].columns) and norm_rows(t["df"]) == norm_rows(t0["df"]) for t in tabs.values())
        if same:
            write_table(f"{scope}.{name}", t0, {"also_in": [{"source_file": tabs[k]["file"], "source_sheet": tabs[k]["sheet"]} for k in MKEYS[1:]]})
        else:
            gap(f"{scope}.{name}", ",".join(tabs), "", "SHARED_MISMATCH", f"table {name} ({sheet}) is not identical in both workbooks; written per model")
            for k, t in tabs.items():
                write_table(f"{k}.{name}", t, {"note": f"{scope} table not identical in both workbooks; kept per model"})

print(f"{len(TABLES)} tables written to {TBL_DIR} | by prefix {dict(Counter(t.split('.')[0] for t in TABLES))}")

In [ ]:
shutil.rmtree(FIG_DIR, ignore_errors=True)       # this module owns figures/; re-running overwrites it
os.makedirs(FIG_DIR)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.edgecolor": "black", "axes.grid": True, "grid.color": "0.85",
                     "grid.linestyle": ":", "legend.frameon": False, "savefig.dpi": 200, "font.family": "DejaVu Sans"})
LINE = {MKEYS[0]: dict(color="black", ls="-", marker="o"), MKEYS[1]: dict(color="0.45", ls="--", marker="s")}
BAR = {MKEYS[0]: dict(color="0.35", edgecolor="black", hatch=""), MKEYS[1]: dict(color="white", edgecolor="black", hatch="///")}
BENCH_LINE = dict(color="0.6", ls=":", marker="^")
FIGS = OrderedDict()

def csv_rows(key, **flt):
    d = pd.read_csv(TABLES[key]["csv"], dtype=str, keep_default_na=False)
    for c, v in flt.items():
        d = d[d[c] == v]
    return d

def fl(s):
    return [float(x) for x in s]

def save_fig(name, fig, caption, tables, series, notes=""):
    png = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(png, bbox_inches="tight")
    plt.close(fig)
    srcs = []
    for t in tables:
        with open(TABLES[t]["sidecar"]) as f:
            sd = json.load(f)
        srcs.append({x: sd[x] for x in ("table", "source_file", "source_sheet", "source_cell_or_row")})
    meta = {"figure": name, "file": png, "caption": caption, "tables": tables, "sources": srcs, "series": series, "notes": notes}
    with open(png[:-4] + ".json", "w") as f:
        json.dump(meta, f, indent=1, ensure_ascii=False)
    FIGS[name] = meta

# F1 population waterfall (shared)
wk = "shared.waterfall" if "shared.waterfall" in TABLES else None
if wk:
    d = csv_rows(wk)
    steps, rem, rgb = list(d["step"]), fl(d["remaining"]), fl(d["remaining_good_bad"])
    fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(steps) + 1.2))
    y = np.arange(len(steps))
    ax.barh(y - 0.2, rem, height=0.4, color="0.35", edgecolor="black", label="All records remaining")
    ax.barh(y + 0.2, rgb, height=0.4, color="white", edgecolor="black", hatch="///", label="GOOD+BAD remaining")
    for yi, v in zip(y, rem):
        ax.text(v, yi - 0.2, f" {v:,.0f}", va="center", fontsize=7)
    ax.set_yticks(y, steps); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_xlabel("Records"); ax.set_title("Population filters: records remaining after each step")
    ax.legend(loc="lower right")
    save_fig("waterfall", fig, "Population waterfall: records and GOOD+BAD records remaining after each population filter, in rule order.",
             [wk], {"step": steps, "remaining": rem, "remaining_good_bad": rgb})
else:
    gap("figure.waterfall", "", "", "FIGURE_NOT_DRAWN", "shared waterfall table not available")

# F2 feature-funnel counts (both models)
fk = [f"{k}.funnel_by_layer" for k in MKEYS if f"{k}.funnel_by_layer" in TABLES]
if len(fk) == len(MKEYS):
    ser = {}
    for k in MKEYS:
        d = csv_rows(f"{k}.funnel_by_layer", source="ALL")
        ser[k] = {"stage": ["candidates"] + list(d["layer"]), "features": [float(d["features_in"].iloc[0])] + fl(d["features_out"])}
    stages = ser[MKEYS[0]]["stage"]
    assert all(ser[k]["stage"] == stages for k in MKEYS), "funnel layers differ between models"
    fig, ax = plt.subplots(figsize=(7, 3.6))
    x, w = np.arange(len(stages)), 0.38
    for j, k in enumerate(MKEYS):
        b = ax.bar(x + (j - 0.5) * w, ser[k]["features"], width=w, label=LABEL[k], **BAR[k])
        ax.bar_label(b, labels=[f"{v:,.0f}" for v in ser[k]["features"]], fontsize=7, padding=1)
    ax.set_xticks(x, ["Candidates" if s == "candidates" else f"after {s}" for s in stages])
    ax.set_ylabel("Features"); ax.set_title("Feature selection funnel (all sources)"); ax.legend()
    save_fig("funnel_counts", fig, "Feature selection funnel: candidate features and features remaining after each layer L0-L5, by model.",
             fk, ser, "candidates = features_in of the first layer row (source ALL)")
else:
    gap("figure.funnel_counts", "", "", "FIGURE_NOT_DRAWN", "funnel_by_layer table missing for a model")

# F3 OOT bad rate by score decile (both models, documented model, overall)
bk = [f"{k}.oot_bands" for k in MKEYS]
if all(t in TABLES for t in bk):
    ser, notes = {}, []
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for k in MKEYS:
        d = csv_rows(f"{k}.oot_bands", model=DOC_MODEL[k], kind="decile", period="overall")
        ser[k] = {"band": fl(d["band"]), "bad_rate": fl(d["bad_rate"]), "mean_pd": fl(d["mean_pd"])}
        ax.plot(ser[k]["band"], ser[k]["bad_rate"], label=LABEL[k], **LINE[k])
        lo, hi = ser[k]["mean_pd"][0], ser[k]["mean_pd"][-1]
        notes.append(f"{k}: band {ser[k]['band'][0]:g} has the {'lowest' if lo < hi else 'highest'} mean PD")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_xticks(ser[MKEYS[0]]["band"])
    ax.set_xlabel("OOT score decile (band as in the OOT report)"); ax.set_ylabel("Bad rate (GOOD+BAD)")
    ax.set_title("OOT bad rate by score decile"); ax.legend()
    save_fig("oot_band_bad_rates", fig, f"OOT bad rate by score decile, Model {DOC_MODEL[MKEYS[0]]} with and without FTS (overall OOT). " + "; ".join(notes) + ".",
             bk, ser, "filter: model = documented model, kind = decile, period = overall")
else:
    gap("figure.oot_band_bad_rates", "", "", "FIGURE_NOT_DRAWN", "oot_bands table missing for a model")

# F4 calibration: mean predicted PD vs observed bad rate by decile (both models)
ck = [f"{k}.oot_calibration_by_decile" for k in MKEYS]
if all(t in TABLES for t in ck):
    ser = {}
    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    for k in MKEYS:
        d = csv_rows(f"{k}.oot_calibration_by_decile", model=DOC_MODEL[k], period="overall")
        ser[k] = {"band": fl(d["band"]), "mean_pd": fl(d["mean_pd"]), "bad_rate": fl(d["bad_rate"])}
        ax.plot(ser[k]["mean_pd"], ser[k]["bad_rate"], label=LABEL[k], **LINE[k])
    top = max(max(ser[k]["mean_pd"] + ser[k]["bad_rate"]) for k in MKEYS)
    ax.plot([0, top], [0, top], color="black", lw=0.8, ls="-.", marker=None, label="Perfect calibration")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel("Mean calibrated PD (decile)"); ax.set_ylabel("Observed bad rate (decile)")
    ax.set_title("OOT calibration by decile"); ax.legend(loc="upper left")
    save_fig("calibration", fig, f"OOT calibration: mean calibrated PD against observed bad rate by decile, Model {DOC_MODEL[MKEYS[0]]} with and without FTS (overall OOT).",
             ck, ser, "filter: model = documented model, period = overall; diagonal = equal PD and bad rate")
else:
    gap("figure.calibration", "", "", "FIGURE_NOT_DRAWN", "oot_calibration_by_decile table missing for a model")

# F5 cut-off curve: approval rate vs approved bad rate (both models + A-Score benchmark)
PCT_RE = re.compile(r"^\s*(-?\d+(?:\.\d+)?)%")
def ascore_cols(cols):
    ar = next((c for c in cols if re.fullmatch(r"A[-_ ]?S(core|CORE)_approval_rate_achieved", c)), None)
    br = next((c for c in cols if re.fullmatch(r"A[-_ ]?S(core|CORE)_approved_bad_rate(_CI)?", c)), None)
    return ar, br
uk = [f"{k}.cutoff_grid_oot_c" for k in MKEYS]
if all(t in TABLES for t in uk):
    ser, tabs_used, notes = {}, list(uk), []
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for k in MKEYS:
        d = csv_rows(f"{k}.cutoff_grid_oot_c")
        ser[k] = {"approval_rate_achieved": fl(d["approval_rate_achieved"]), "approved_bad_rate": fl(d["approved_bad_rate"])}
        ax.plot(ser[k]["approval_rate_achieved"], ser[k]["approved_bad_rate"], label=LABEL[k], **LINE[k])
    a_src = None
    for k in MKEYS:
        t = f"{k}.cutoff_summary"
        if t not in TABLES:
            continue
        d = csv_rows(t)
        ar, br = ascore_cols(d.columns)
        if ar and br:
            if a_src is None:
                ys = [float(PCT_RE.match(v).group(1)) / 100 for v in d[br]]
                ser["A_SCORE"] = {"approval_rate_achieved": fl(d[ar]), "approved_bad_rate": ys, "from": f"{t}: {ar}, {br}"}
                ax.plot(ser["A_SCORE"]["approval_rate_achieved"], ys, label="A-Score (current)", **BENCH_LINE)
                tabs_used.append(t); a_src = k
        else:
            gap(f"{k}.cutoff_summary.a_score_approved_bad_rate", k, "Risk tranches / cut-off", "NOT_IN_EVIDENCE",
                f"{k} evidence has no A-Score approval rate / approved bad rate by cut-off (cut-off workbook sheet vs_benchmarks not copied)")
    if a_src:
        notes.append(f"A-Score curve from {a_src} cut-off summary; its approved bad rate is the value printed in the report (2 decimals of a percent)")
    else:
        gap("figure.cutoff_curve.a_score", "", "Risk tranches / cut-off", "NOT_IN_EVIDENCE", "no A-Score cut-off curve in either workbook")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel("Approval rate (previously approved applicants, OOT)"); ax.set_ylabel("Bad rate of approved (GOOD+BAD)")
    ax.set_title("Cut-off curve: approval rate vs approved bad rate"); ax.legend()
    save_fig("cutoff_curve", fig, "Cut-off curve on OOT: approval rate against bad rate of approved applicants for Model C with and without FTS and the current A-Score. "
             "Approval rates are relative to previously approved applicants.", tabs_used, ser, "; ".join(notes))
else:
    gap("figure.cutoff_curve", "", "", "FIGURE_NOT_DRAWN", "cutoff_grid_oot_c table missing for a model")

print(f"{len(FIGS)} figures: {list(FIGS)}")

In [ ]:
STRUCTURAL = {"meta", "index", "inventory_reports"}       # evidence-pack bookkeeping, not quotable content
if not MAP_IS_APPROVED:
    gap("template_map_approved.xlsx", "", "", "APPROVED_MAP_MISSING",
        f"{MAP_APPROVED} not found; used {MAP_PATH} (the map agreed on 2026-11-26 with its amendments)")
MAP = pd.read_excel(note_read(MAP_PATH), sheet_name="map")
for r in MAP.itertuples():
    sec = str(r.template_heading)
    ev_lines = [] if pd.isna(r.evidence_sheets) else str(r.evidence_sheets).split("\n")
    for line in ev_lines:
        m = re.match(r"^\s*([^\s\[]+)", line)
        if not m:
            continue
        expr = m.group(1)
        for k in MKEYS:
            s = pick_sheet(k, expr)
            if s is None:
                gap(f"{k}.{slug(expr)}", k, sec, "MISSING_EVIDENCE", f"map requests evidence sheet {expr}; not in the {k} workbook", sec)
            elif STATUS[k].get(s) in ("EMPTY", "MISSING"):
                gap(f"{k}.{slug(s)}", k, sec, "EMPTY", f"map requests evidence sheet {s}; it is empty in the {k} workbook", sec)
            elif s not in STRUCTURAL and s not in USED[k]:
                gap(f"{k}.{slug(s)}", k, sec, "NOT_EXTRACTED", f"map requests evidence sheet {s}; no fact or table was built from it", sec)
    if r.status in ("PARTIAL", "GAP") and not pd.isna(r.what_is_missing):
        for line in str(r.what_is_missing).split("\n"):
            if not line.strip():
                continue
            gt = "HUMAN_INPUT_OPEN" if line.startswith("human input") else ("MISSING_EVIDENCE" if line.startswith("evidence sheet") else "NOT_IN_EVIDENCE")
            gap(f"section.{slug(sec)}", "both", sec, gt, line, sec)

GAPS_DF = pd.DataFrame(GAPS, columns=["key", "model_key", "section", "gap_type", "detail", "requested_by"]).drop_duplicates().reset_index(drop=True)
with open(OUT_FACTS, "w") as f:
    json.dump(FACTS, f, indent=1, ensure_ascii=False)
GAPS_DF.to_csv(OUT_GAPS, index=False)
print(f"wrote {OUT_FACTS} ({len(FACTS)} facts) and {OUT_GAPS} ({len(GAPS_DF)} rows: {dict(Counter(GAPS_DF['gap_type']))})")

In [ ]:
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "pass": bool(ok), "detail": str(detail)[:800]})

SRC_FIELDS = ("source_file", "source_sheet", "source_cell_or_row")
KEY_RE = re.compile(r"^(" + "|".join(map(re.escape, MKEYS)) + r"|shared|diff)\.[a-z0-9_]+\.[a-z0-9_.]+$")
bad_src = [k for k, e in FACTS.items() if not all(e.get(x) for x in SRC_FIELDS) or "value" not in e or "display" not in e]
check("every fact has value, display, source_file, source_sheet, source_cell_or_row", not bad_src, bad_src[:10])
check("every fact key is <model key>|shared|diff.<area>.<item>", all(KEY_RE.match(k) for k in FACTS), [k for k in FACTS if not KEY_RE.match(k)][:10])
check("every fact source is a Step 1 evidence workbook", all(e["source_file"] in EV_WB.values() for e in FACTS.values()))

# re-read every referenced cell from the evidence workbooks (fresh load) and compare with the fact value
WBF = {p: openpyxl.load_workbook(p, data_only=True) for p in EV_WB.values()}
def cell_val(file, sheet, ref):
    return native(WBF[file][sheet][ref].value)
mism = []
for key, e in FACTS.items():
    for loc in [e] + e.get("also_in", []):
        if cell_val(loc["source_file"], loc["source_sheet"], loc["source_cell_or_row"]) != e["value"]:
            mism.append(f"{key} @ {loc['source_sheet']}!{loc['source_cell_or_row']}")
check("no value computed: every fact equals its evidence cell (shared/diff: in both workbooks)", not mism, mism[:10])
check("inputs read: only config, the evidence workbooks and the template map",
      set(READS) <= {os.path.realpath(p) for p in [CFG_PATH, MAP_PATH] + list(EV_WB.values())}, sorted(set(READS)))
shared_keys = [k for k in FACTS if k.split(".")[0] in ("shared", "diff")]
check("shared/diff facts are identical in both workbooks; differences listed in facts_gaps.csv",
      all(len(FACTS[k].get("also_in", [])) == len(MKEYS) - 1 for k in shared_keys)
      and all(g["gap_type"] == "SHARED_MISMATCH" for g in GAPS if "not identical" in g["detail"]), f"{len(shared_keys)} shared/diff facts")
G3 = [k for k, e in FACTS.items() if isinstance(e["value"], float) and not P_RE.search(k)
      and set(re.split(r"[._]", k)) & {"gini", "dgini"} and not set(re.split(r"[._]", k)) & PCT_T]
PC = [k for k, e in FACTS.items() if isinstance(e["value"], float) and not P_RE.search(k) and "learning" not in k
      and (set(re.split(r"[._]", k)) & PCT_T)]
check("display: Gini 3 decimals, rates % with 1 decimal, counts with thousands separators",
      all(re.fullmatch(r"-?\d+\.\d{3}", FACTS[k]["display"]) for k in G3) and all(re.fullmatch(r"-?\d+\.\d%", FACTS[k]["display"]) for k in PC)
      and all("," in FACTS[k]["display"] for k, e in FACTS.items() if isinstance(e["value"], int) and not isinstance(e["value"], bool)
              and abs(e["value"]) >= 1000 and not P_RE.search(k) and not set(re.split(r"[._]", k)) & PCT_T),
      f"{len(G3)} Gini facts, {len(PC)} rate facts")
req = ["shared.counts.dev_records", "fts_c.oot.gini_overall", "nofts_c.oot.gini_ci_low", "diff.cutoff.p95.net_bads"]
check("example keys from the request exist", all(k in FACTS for k in req), [k for k in req if k not in FACTS])

# tables: csv equals the evidence block; sidecar has the source fields
tbad = []
for key, t in TABLES.items():
    d = pd.read_csv(t["csv"], dtype=str, keep_default_na=False)
    with open(t["sidecar"]) as f:
        sd = json.load(f)
    if list(d.columns) != list(t["tab"]["df"].columns) or norm_rows(d) != norm_rows(t["tab"]["df"]) or not all(sd.get(x) for x in SRC_FIELDS):
        tbad.append(key)
check("every table csv equals its evidence block and its sidecar names the source", not tbad and len(TABLES) > 0, tbad[:10])

# figures: plotted series equal the evidence values of the same rows (independent path: workbook block, not the csv)
def wb_series(tkey, cols, **flt):
    df = TABLES[tkey]["tab"]["df"]
    for c, v in flt.items():
        df = df[df[c].astype(str) == v]
    return {c: [float(native(x)) for x in df[c]] for c in cols}
fbad = []
f = FIGS.get("waterfall")
if f and wb_series("shared.waterfall", ["remaining", "remaining_good_bad"]) != {c: f["series"][c] for c in ("remaining", "remaining_good_bad")}:
    fbad.append("waterfall")
f = FIGS.get("funnel_counts")
if f:
    for k in MKEYS:
        w = wb_series(f"{k}.funnel_by_layer", ["features_in", "features_out"], source="ALL")
        if [w["features_in"][0]] + w["features_out"] != f["series"][k]["features"]:
            fbad.append(f"funnel_counts {k}")
for name, tname, flt, cols in [("oot_band_bad_rates", "oot_bands", {"kind": "decile", "period": "overall"}, ["band", "bad_rate"]),
                               ("calibration", "oot_calibration_by_decile", {"period": "overall"}, ["mean_pd", "bad_rate"]),
                               ("cutoff_curve", "cutoff_grid_oot_c", {}, ["approval_rate_achieved", "approved_bad_rate"])]:
    f = FIGS.get(name)
    if not f:
        continue
    for k in MKEYS:
        fl_ = dict(flt, **({"model": DOC_MODEL[k]} if name != "cutoff_curve" else {}))
        if wb_series(f"{k}.{tname}", cols, **fl_) != {c: f["series"][k][c] for c in cols}:
            fbad.append(f"{name} {k}")
f = FIGS.get("cutoff_curve")
if f and "A_SCORE" in f["series"]:
    t = next(x for x in f["tables"] if x.endswith(".cutoff_summary"))
    ar, br = ascore_cols(list(TABLES[t]["tab"]["df"].columns))
    df = TABLES[t]["tab"]["df"]
    if [float(native(x)) for x in df[ar]] != f["series"]["A_SCORE"]["approval_rate_achieved"] or \
       [float(PCT_RE.match(str(x)).group(1)) / 100 for x in df[br]] != f["series"]["A_SCORE"]["approved_bad_rate"]:
        fbad.append("cutoff_curve A_SCORE")
fig_files_ok = all(os.path.exists(m["file"]) and os.path.exists(m["file"][:-4] + ".json") and m["caption"] and m["sources"] for m in FIGS.values())
check("figure data equal the source tables (re-checked against the evidence); png + caption/source json per figure",
      not fbad and fig_files_ok and len(FIGS) == 5, f"{list(FIGS)} | mismatches {fbad}")
check("every evidence sheet requested by the template map is extracted or listed in facts_gaps.csv",
      all(any(g["key"].endswith(slug(s)) for g in GAPS) or s in USED[k] or s in STRUCTURAL
          for r in MAP.itertuples() if not pd.isna(r.evidence_sheets)
          for line in str(r.evidence_sheets).split("\n") if re.match(r"^\s*([^\s\[]+)", line)
          for k in MKEYS for s in [pick_sheet(k, re.match(r"^\s*([^\s\[]+)", line).group(1))] if s))
check("facts_gaps.csv written with key, model_key, section, gap_type, detail, requested_by",
      list(pd.read_csv(OUT_GAPS).columns) == ["key", "model_key", "section", "gap_type", "detail", "requested_by"])
check("inputs unchanged during the run (sha256)", sha256_file(CFG_PATH) == INPUT_SHA["config"] and sha256_file(MAP_PATH) == INPUT_SHA["map"]
      and all(sha256_file(v) == INPUT_SHA[f"evidence_{k}"] for k, v in EV_WB.items()))

COUNTS = {"facts": len(FACTS), "facts_by_prefix": dict(Counter(k.split(".")[0] for k in FACTS)), "tables": len(TABLES), "figures": len(FIGS),
          "gaps": len(GAPS_DF), "gaps_by_type": dict(Counter(GAPS_DF["gap_type"])), "notes": len(NOTES)}
os.makedirs(os.path.dirname(RUN_SUM), exist_ok=True)
summary = {"module": MODULE, "run_time_utc": RUN_TIME, "git_commit": GIT, "git_branch": BRANCH, "config_hash": CFG_SHA,
           "inputs": {"evidence": EV_WB, "template_map": MAP_PATH, "template_map_is_approved_file": MAP_IS_APPROVED, "sha256": INPUT_SHA},
           "outputs": {"facts": OUT_FACTS, "gaps": OUT_GAPS, "tables_dir": TBL_DIR, "figures_dir": FIG_DIR},
           "counts": COUNTS, "key_facts": {k: FACTS[k]["display"] for k in req if k in FACTS}, "notes": NOTES[:50],
           "checks": CHECKS, "pass_fail": "PASS" if all(c["pass"] for c in CHECKS) else "FAIL"}
with open(RUN_SUM, "w") as f:
    json.dump(summary, f, indent=1, ensure_ascii=False)

SNAP_AFTER = snapshot(WATCH)
changed = sorted(p for p in set(SNAP_BEFORE) | set(SNAP_AFTER) if SNAP_BEFORE.get(p) != SNAP_AFTER.get(p))
own = lambda p: p in (OUT_FACTS, OUT_GAPS, RUN_SUM) or p.startswith(TBL_DIR + os.sep) or p.startswith(FIG_DIR + os.sep)
write_ok = all(own(p) for p in changed)
for c in CHECKS:
    print(("PASS  " if c["pass"] else "FAIL  ") + c["check"] + ("" if c["pass"] else f" | {c['detail']}"))
print(("PASS  " if write_ok else "FAIL  ") + "only own outputs written under <output_folder>/ (before/after snapshot)")
if not write_ok:
    print("unexpected changes:", [p for p in changed if not own(p)][:20])
print(f"{COUNTS} | run summary {RUN_SUM}")
for c in CHECKS:
    assert c["pass"], f"acceptance failed: {c['check']} | {c['detail']}"
assert write_ok, "acceptance failed: files written outside this module's outputs"
print("All acceptance checks PASS")